In [0]:
-- SET pipelines.clusterShutdown.delay=2s
-- SET datasets.path=dbfs:/mnt/demo-datasets/bookstore;

In [0]:
CREATE OR REFRESH STREAMING LIVE TABLE books_bronze
COMMENT "The raw books data from CDC feed"
AS SELECT * FROM cloud_files("${datasets.path}/books-cdc", "json")

In [0]:
CREATE OR REFRESH STREAMING LIVE TABLE books_silver;

APPLY CHANGES INTO LIVE.books_silver
  FROM LIVE.books_bronze
  KEYS(book_id)
  APPLY AS DELETE WHEN row_status = 'DELETE'
  SEQUENCE BY row_time
  COLUMNS * EXCEPT (row_status, row_time)

In [0]:
CREATE LIVE TABLE author_counts_state
 COMMENT "Num of books per author"
AS SELECT author, COUNT(*) AS books_count, current_timestamp() updated_time
  FROM LIVE.books_silver
  GROUP BY author

In [0]:
CREATE LIVE VIEW books_sales
  AS SELECT b.title, o.quantity
    FROM (
      SELECT *, explode(books) as book
      FROM LIVE.orders_cleaned) o
    INNER JOIN LIVE.books_silver b
    ON o.book.book_id = b.book_id;